In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!mkdir /root/.config/kaggle/
!cp /content/drive/MyDrive/"פרויקט גמר"/kaggle.json /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json

In [ ]:
# Creating a dataset directory
!mkdir ./datasets
!mkdir ./datasets/OpenForensicsV1

# download the dataset from Kaggle and unzip it
!kaggle datasets download manjilkarki/deepfake-and-real-images -p ./datasets/OpenForensicsV1
!unzip ./datasets/OpenForensicsV1/*.zip  -d ./datasets/OpenForensicsV1

!kaggle datasets download birdy654/cifake-real-and-ai-generated-synthetic-images -p ./datasets/cifake
!unzip ./datasets/cifake/*.zip  -d ./datasets/cifake

# Ensemble Evaluation on **CiFAKE** and **OpenForensics‑V1**
This notebook loads four TensorFlow models per dataset, ensembles their predictions, and reports accuracy, precision, recall, F1‑score, AUROC, and a confusion matrix.

**Prerequisites**  
• You are running this inside the provided Docker container with GPU support.  
• `TF_FORCE_GPU_ALLOW_GROWTH=true` is set to avoid OOM crashes.  
• Datasets are mounted at `/data/cifake` and `/data/openforensicsv1`.  
• Trained model files are inside `/workspace/models/{cifake,openforensics}`.

In [4]:
import tensorflow as tf, sys, os, json, random, numpy as np
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow :", tf.__version__)
print("cuDNN      :", tf.sysconfig.get_build_info()["cudnn_version"])
print("GPU list   :", tf.config.list_physical_devices('GPU'))
assert gpus, "⚠️  No GPU found – make sure container runs with `--gpus all`"


TensorFlow version: 2.18.0
TensorFlow : 2.18.0
cuDNN      : 9
GPU list   : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [46]:
from pathlib import Path
import os
import json

# Adjust these if your mount points differ
DATA_ROOT = Path(os.getcwd() + '/datasets')
MODEL_ROOT = Path('/content/drive/MyDrive/פרויקט גמר/ensemble/models')
print(DATA_ROOT)
print(MODEL_ROOT)

DATASETS = {
    "cifake": {
        "root":  DATA_ROOT / "cifake" / "test",
        "alts":  dict(real=["REAL", "real"], fake=["FAKE", "fake"]),
        "models": sorted((MODEL_ROOT / "CIFAKE").glob("*.keras")),
    },
    "openforensics": {
        "root":  DATA_ROOT / "OpenForensicsV1" / "Dataset" / "Test",
        "alts":  dict(real=["real", "genuine", "original"],
                      fake=["fake", "manipulated", "ai"]),
        "models": sorted((MODEL_ROOT / 'OpenForensics').glob("*.keras")),
    },
}

print(json.dumps({k: {'n_models': len(v['models'])} for k, v in DATASETS.items()}, indent=2))


/content/datasets
/content/drive/MyDrive/פרויקט גמר/ensemble/models
{
  "cifake": {
    "n_models": 4
  },
  "openforensics": {
    "n_models": 4
  }
}


In [56]:
# ── Vectorised GPU-bound weight calculator (helpers-based) ────────────────
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np, tensorflow as tf
from tqdm import tqdm

def compute_dataset_weights(
    ds_key: str,
    batch: int  = 512,        # L4 sweet-spot
    power: float = 2.0,
    min_weight: float = 0.05,
    threshold: float = 0.5,
) -> dict[str, float]:

    cfg = DATASETS[ds_key]

    # 1️⃣ gather paths & labels ---------------------------------------------
    fake  = _collect_class(cfg["root"], cfg["alts"]["fake"])
    real  = _collect_class(cfg["root"], cfg["alts"]["real"])
    str_paths = [str(p) for p in fake + real]                 # PosixPath → str
    labels    = np.array([0]*len(fake) + [1]*len(real), dtype=np.int8)

    # 2️⃣ load models, group by expected input ------------------------------
    groups: dict[str, list[tuple[str, tf.keras.Model]]] = {"srm": [], 8192: [], 16384: []}
    for p in cfg["models"]:
        m   = safe_load(p, compile=False)
        key = "srm" if len(m.input_shape) == 4 else m.input_shape[1]   # 8192/16384
        groups[key].append((p.stem, m))

    # 3️⃣ build tf.data pipeline (CPU decode → GPU prefetch) ----------------
    dataset = build_ds(str_paths, batch)     # uses load_and_prep_tf inside

    # 4️⃣ accumulate predictions -------------------------------------------
    y_hat = {name: [] for g in groups.values() for name, _ in g}

    for imgs in tqdm(dataset, desc=f"eval {ds_key}", unit="batch", unit_scale=batch):

        # SRM-CNN
        if groups["srm"]:
            preds = [m(imgs, training=False)[:, 0] for _, m in groups["srm"]]
            for (name, _), p in zip(groups["srm"], preds):
                y_hat[name].extend(p.numpy())

        # DCT-128
        if groups[8192]:
            feats128 = batch_encode_dct_gpu(imgs, 128)                 # (B, 8192)
            preds128 = [m(feats128, training=False)[:, 0] for _, m in groups[8192]]
            for (name, _), p in zip(groups[8192], preds128):
                y_hat[name].extend(p.numpy())

        # DCT-256
        if groups[16384]:
            feats256 = batch_encode_dct_gpu(imgs, 256)                 # (B,16384)
            preds256 = [m(feats256, training=False)[:, 0] for _, m in groups[16384]]
            for (name, _), p in zip(groups[16384], preds256):
                y_hat[name].extend(p.numpy())

    # 5️⃣ accuracy + AUC → weights -----------------------------------------
    accs, aucs, names = [], [], []
    for name, preds in y_hat.items():
        preds = np.asarray(preds)
        accs.append(accuracy_score(labels, preds > threshold))
        try:
            aucs.append(roc_auc_score(labels, preds))
        except ValueError:                      # single-class fallback
            aucs.append(0.5)
        names.append(name)
        print(f"{name:18s} Acc={accs[-1]:.3f}  AUC={aucs[-1]:.3f}")

    combined = 0.5*np.array(accs) + 0.5*np.array(aucs)
    weights  = np.power(combined, power)
    weights  = np.maximum(weights, min_weight)
    weights /= weights.sum()

    return dict(zip(names, map(float, weights)))


In [ ]:
# --- Pre‑computed model weights for weighted averaging ---
# Keys must match the *stem* (filename without extension) of each .keras model file
model_weights = {
    "SRM-CNN-L256": 0.2094,
    "DCT-AE-L256": 0.296,
    "SRM-CNN-L128": 0.2054,
    "DCT-AE-L128": 0.2892,
}
total_w = sum(model_weights.values())
print("Loaded model_weights – total =", total_w)


Loaded model_weights – total = 1.0


In [48]:
# ── SRM filter constants ---------------------------------------------------
IMG_SIZE = (256, 256)

SRM_FILTERS = np.array([
    # F1  Laplacian-High Boost
    [[[ 0,  0, -1,  0,  0],
      [ 0, -1,  2, -1,  0],
      [-1,  2,  4,  2, -1],
      [ 0, -1,  2, -1,  0],
      [ 0,  0, -1,  0,  0]]],
    # F2  Edge & Noise Enhancer
    [[[-1,  2, -2,  2, -1],
      [ 2, -6,  8, -6,  2],
      [-2,  8,-12,  8, -2],
      [ 2, -6,  8, -6,  2],
      [-1,  2, -2,  2, -1]]],
    # F3  Diagonal Residual
    [[[ 2, -1,  0, -1,  2],
      [-1, -2,  3, -2, -1],
      [ 0,  3,  0,  3,  0],
      [-1, -2,  3, -2, -1],
      [ 2, -1,  0, -1,  2]]],
    # F4  Vertical Edge
    [[[ 0,  0,  0,  0,  0],
      [ 1, -2,  1, -2,  1],
      [ 0,  0,  0,  0,  0],
      [-1,  2, -1,  2, -1],
      [ 0,  0,  0,  0,  0]]],
    # F5  High-frequency Noise
    [[[ 1, -4,  6, -4,  1],
      [-4, 16,-24, 16, -4],
      [ 6,-24, 36,-24,  6],
      [-4, 16,-24, 16, -4],
      [ 1, -4,  6, -4,  1]]],
], dtype=np.float32)

# (5,5,1,5) for tf.nn.conv2d
SRM_FILTERS_TF = tf.constant(np.transpose(SRM_FILTERS, (2, 3, 1, 0)),
                             dtype=tf.float32)


# ── Graph-safe SRM filter --------------------------------------------------
@tf.function(jit_compile=True)
def apply_srm_filters_tf(image: tf.Tensor) -> tf.Tensor:
    """
    image  : (H,W,3)  or  (B,H,W,3), float32 in [0,1]
    returns: (H,W,15) or  (B,H,W,15)  float32
    """
    rank = image.shape.rank
    if rank == 3:                # add batch dim for conv2d
        image = image[None, ...]
        squeeze = True
    else:
        squeeze = False

    channels = tf.split(image, 3, axis=-1)          # R,G,B
    maps = [tf.nn.conv2d(c, SRM_FILTERS_TF,
                         strides=1, padding='SAME') for c in channels]
    out = tf.concat(maps, axis=-1)                  # 15 channels

    return out[0] if squeeze else out


# ── Helper for tf.data pipelines ------------------------------------------
def load_and_prep_tf(path: tf.Tensor) -> tf.Tensor:
    """tf.string → SRM-filtered tensor (256,256,15), all in TF graph."""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)       # or decode_png
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    srm = apply_srm_filters_tf(img)                   # (256,256,15)
    srm.set_shape((256, 256, 15))                     # static shape hint
    return srm


# ── Dataset builder --------------------------------------------------------
def build_ds(path_list: list[str], batch: int) -> tf.data.Dataset:
    """
    Creates a pipeline with CPU decode/resize and overlapped GPU prefetch.
    """
    return (tf.data.Dataset.from_tensor_slices(path_list)
            .map(load_and_prep_tf,  num_parallel_calls=tf.data.AUTOTUNE)
            .batch(batch)
            .prefetch(tf.data.AUTOTUNE))


In [53]:
@tf.function(jit_compile=True)        # → XLA kernel, stays on GPU
def _dct_1d_gpu(x):
    """DCT-II via even extension + cuFFT (GPU only).  x: [..., N]"""
    N  = tf.shape(x)[-1]
    x2 = tf.concat([x, x[..., ::-1]], axis=-1)      # even extension
    X  = tf.signal.rfft(x2)                         # cuFFT on GPU
    k  = tf.cast(tf.range(N), x.dtype)
    factor = tf.exp(-1j * np.pi * k / (2*N))
    return tf.math.real(X[..., :N] * factor) * 2.0  # DCT-II

@tf.function(jit_compile=True)
def _dct_batch(vecs):
    """
    vecs: (B, N) float32.   Returns DCT-II of every row, L2-normalised.
    Runs entirely on GPU (cuFFT).
    """
    N  = tf.shape(vecs)[-1]
    # Even extension
    x2 = tf.concat([vecs, vecs[:, ::-1]], axis=-1)        # (B, 2N)
    X  = tf.signal.rfft(x2)                               # cuFFT
    k  = tf.cast(tf.range(N)[None, :], vecs.dtype)
    factor = tf.exp(-1j * np.pi * k / (2*N))
    dct = tf.math.real(X[:, :N] * factor) * 2.0           # DCT-II
    return tf.math.l2_normalize(dct, axis=1)

@tf.function(jit_compile=True)
def batch_encode_dct_gpu(imgs, latent_size):          # imgs: (B,256,256,15)
    # 1. encoder pass (8×8×C) – already in your code
    lat = ENCODERS[latent_size](imgs, training=False)         # (B,8,8,C)

    # 2. flatten per sample
    flat = tf.reshape(lat, (tf.shape(imgs)[0], -1))          # (B,8192/16384)

    # 3. orthonormal DCT-II (GPU/cuFFT via XLA)
    dct  = tf.signal.dct(flat, type=2, norm='ortho')

    # 4. ℓ2-normalise
    return tf.linalg.l2_normalize(dct, axis=1)

In [50]:
# --------------------------------------------------------------
#  Dual-latent autoencoder loader  (read-only-mount safe)
# --------------------------------------------------------------
from pathlib import Path
import shutil, tempfile
import tensorflow as tf
from tensorflow.keras.models import load_model, Model

AE_DIR     = MODEL_ROOT / "ae"                # contains autoencoder_L128/256.keras
IMG_SIZE   = (256, 256)

def _safe_load(path: Path):
    """Copy `path` to a writable tmp dir on first use, then load."""
    tmp = Path(tempfile.gettempdir()) / path.name
    if not tmp.exists():
        shutil.copy(path, tmp)                # tmp is always RW
    return load_model(tmp, compile=False)

ENCODERS = {}                                 # {128: encoder128, 256: encoder256}
for latent_size in (128, 256):
    ae_path = AE_DIR / f"autoencoder_L{latent_size}.keras"
    if not ae_path.exists():
        print(f"⚠️  Autoencoder L{latent_size} missing → {ae_path}")
        continue
    autoencoder = _safe_load(ae_path)
    encoder = Model(autoencoder.input,
                    autoencoder.get_layer("encoder_output").output)
    encoder.trainable = False
    ENCODERS[latent_size] = encoder
    print(f"✓  Encoder L{latent_size} ready → {encoder.output_shape}")

def load_and_prep(path: Path) -> tf.Tensor:
    """JPEG → float32 → resize → SRM-15ch (256,256,15)."""
    img = tf.io.read_file(str(path))
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return apply_srm_filters_tf(img[tf.newaxis, ...])[0]

def encode_dct(img_tensor: tf.Tensor, latent_size: int = 128) -> tf.Tensor:
    """
    SRM image → encoder → flatten → GPU-DCT → L2-norm (all on GPU).
    img_tensor: (256,256,15) float32, NOT batched.
    """
    encoder = ENCODERS[latent_size]
    latent  = encoder(img_tensor[None, ...], training=False)[0]   # (8,8,L)
    vec     = tf.reshape(latent, [-1])
    dct     = _dct_1d_gpu(vec)
    return tf.math.l2_normalize(dct, axis=0)


✓  Encoder L128 ready → (None, 8, 8, 128)
✓  Encoder L256 ready → (None, 8, 8, 256)


In [51]:
# --- Imports (only new ones) ---
from tqdm import tqdm                      # nice progress bar
import functools                           # for partial
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt


# ---------- Config & path helpers ----------

def _collect_class(root: Path, alternatives: list[str]) -> list[Path]:
    """Return first existing dir’s images (recursively)."""
    for name in alternatives:
        p = root / name
        if p.exists():
            return sorted(p.rglob("*.jp*")) + sorted(p.rglob("*.png"))
    return []

_TMP_DIR = Path(tempfile.gettempdir())          # always writable

def safe_load(path: Path, compile=False) -> tf.keras.Model:
    """
    Load a Keras model from `path`, copying it to /tmp if the original
    location is read-only.  Subsequent calls re-use the cached copy.
    """
    path = Path(path)
    tmp  = _TMP_DIR / path.name
    if not tmp.exists():
        shutil.copy(path, tmp)                  # one-time copy
    return tf.keras.models.load_model(tmp, compile=compile)

# ---------- Core: latent-aware + SRM-CNN ensemble (vectorised) -----------
def ensemble_predict(
    ds_key: str,
    batch: int = 512,          # fits in 3 GB on an L4
    threshold: float = 0.5,
    model_weights: dict[str, float] | None = None,
):
    cfg = DATASETS[ds_key]

    # 1️⃣  gather image paths & labels  -------------------------------------
    fake  = _collect_class(cfg["root"], cfg["alts"]["fake"])
    real  = _collect_class(cfg["root"], cfg["alts"]["real"])
    str_paths = [str(p) for p in fake + real]              # PosixPath → str
    labels    = np.array([0] * len(fake) + [1] * len(real), dtype=np.int8)

    # 2️⃣  load & group models once -----------------------------------------
    groups: dict[str, list[tuple[float, tf.keras.Model]]] = {"srm": [], 8192: [], 16384: []}
    for p in cfg["models"]:
        w = (model_weights or {}).get(p.stem)
        if w is None:
            continue
        m   = safe_load(p, compile=False)
        key = "srm" if len(m.input_shape) == 4 else m.input_shape[1]
        groups[key].append((w, m))

    scores = np.zeros(len(str_paths), np.float32)
    w_sum  = np.zeros(len(str_paths), np.float32)

    # 3️⃣  tf.data pipeline (CPU decode → GPU prefetch) ----------------------
    dataset = build_ds(str_paths, batch)   # uses load_and_prep_tf inside

    # 4️⃣  stream a batch through every model --------------------------------
    offset = 0
    for imgs in tqdm(dataset, desc=f"stream {ds_key}", unit="batch", unit_scale=batch):
        bsz = imgs.shape[0]

        # --- SRM-CNN models -------------------------------------------------
        if groups["srm"]:
            preds = sum(w * m(imgs, training=False)[:, 0] for w, m in groups["srm"])
            scores[offset:offset+bsz] += preds.numpy()
            w_sum[offset:offset+bsz]  += sum(w for w, _ in groups["srm"])

        # --- DCT-128 (8192-D) models ---------------------------------------
        if groups[8192]:
            feats128 = batch_encode_dct_gpu(imgs, 128)           # (B, 8192)
            preds128 = sum(w * m(feats128, training=False)[:, 0] for w, m in groups[8192])
            scores[offset:offset+bsz] += preds128.numpy()
            w_sum[offset:offset+bsz]  += sum(w for w, _ in groups[8192])

        # --- DCT-256 (16384-D) models --------------------------------------
        if groups[16384]:
            feats256 = batch_encode_dct_gpu(imgs, 256)           # (B,16384)
            preds256 = sum(w * m(feats256, training=False)[:, 0] for w, m in groups[16384])
            scores[offset:offset+bsz] += preds256.numpy()
            w_sum[offset:offset+bsz]  += sum(w for w, _ in groups[16384])

        offset += bsz

    # 5️⃣  metrics -----------------------------------------------------------
    weighted = scores / w_sum
    y_pred   = (weighted > threshold).astype(np.int8)

    acc  = accuracy_score(labels, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, y_pred,
                                                       average="binary",
                                                       zero_division=0)
    auc  = roc_auc_score(labels, weighted)
    cm   = confusion_matrix(labels, y_pred)

    print(classification_report(labels, y_pred, target_names=["Fake", "Real"]))
    print(f"Accuracy={acc:.4f} | F1={f1:.4f} | AUROC={auc:.4f}")

    plt.figure(figsize=(4,4))
    plt.imshow(cm, cmap="Blues")
    plt.title(f"{ds_key.upper()} ensemble"); plt.colorbar()
    plt.xticks([0,1], ["Fake","Real"]); plt.yticks([0,1], ["Fake","Real"])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i,j], ha="center", va="center", weight="bold")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.show()

    return dict(acc=acc, precision=prec, recall=rec, f1=f1, auc=auc)


In [ ]:
model_weights_cifake = compute_dataset_weights("cifake", batch=64, threshold=0.45)
print("Computed weights –", model_weights_cifake)

# then pass to your (streaming) ensemble
results_cifake = ensemble_predict(
    "cifake",
    batch     = 64,
    threshold = 0.45,
    model_weights   = model_weights_cifake
)

eval cifake:  73%|███████▎  | 14528/20032 [02:10<00:48, 114.00batch/s]

In [ ]:
results = {}
for key in DATASETS:
    print('\n' + '='*30)
    print(f'Evaluating ensemble on {key.upper()}')
    results[key] = ensemble_predict(key)

import pandas as pd
import json, pprint
df = pd.DataFrame(results).T.round(4)
print('\nSummary:'); display(df)
